In [31]:
import pandas as pd
import re

In [32]:
df = pd.read_csv("../data/email_evaluation_dataset_Abhay-Mulani.csv")

df.head()

,email_text,expected_action,expected_tone
0,Reminder: Project review meeting scheduled tom...,notify,neutral
1,Please find attached the invoice for your Augu...,respond,urgent
2,Thank you for registering for our webinar. No ...,ignore,polite
3,Can you please share the updated design docume...,respond,urgent
4,This is to inform you that the office will rem...,notify,neutral


In [33]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_email"] = df["email_text"].apply(clean_text)


In [34]:
IMPORTANT_KEYWORDS = ["urgent", "submit", "deadline"]
THANK_YOU_KEYWORDS = ["thank you", "thanks", "thankyou"]

def email_assistant(email):
    # urgent emails
    for word in IMPORTANT_KEYWORDS:
        if word in email:
            return "notify", "urgent"
    
    # polite / thank-you emails
    for word in THANK_YOU_KEYWORDS:
        if word in email:
            return "ignore", "polite"
    
    # default normal emails
    return "respond", "neutral"


In [35]:
df[["predicted_action", "predicted_tone"]] = (
    df["clean_email"]
    .apply(email_assistant)
    .apply(pd.Series)
)


In [36]:
df["action_correct"] = df["predicted_action"] == df["expected_action"]
df["tone_correct"] = df["predicted_tone"] == df["expected_tone"]


In [37]:
action_accuracy = df["action_correct"].mean() * 100
tone_accuracy = df["tone_correct"].mean() * 100

action_accuracy, tone_accuracy


(41.333333333333336, 65.33333333333333)

In [38]:
errors = df[df["action_correct"] == False]

errors[[
    "email_text",
    "expected_action",
    "predicted_action"
]].head(10)


,email_text,expected_action,predicted_action
0,Reminder: Project review meeting scheduled tom...,notify,respond
4,This is to inform you that the office will rem...,notify,respond
5,Your OTP for account login is 482913. Do not s...,notify,respond
7,Monthly newsletter: Top tech trends you should...,ignore,respond
12,Team lunch planned this Friday at 1 PM.,notify,respond
13,Your password was changed successfully.,notify,respond
14,Limited-time offer! Get 50% off on premium plans.,ignore,respond
16,This email is to acknowledge receipt of your a...,ignore,respond
17,Server maintenance scheduled tonight from 12 A...,notify,respond
18,Urgent: Action required to avoid service disru...,respond,notify


In [39]:
df.to_csv(
    "../data/milestone2_output_AbhayMulani.csv",
    index=False
)


### Reflection

**Which type of emails were hardest to classify?**  
Emails that required action but did not contain explicit keywords like "urgent" or "submit" were the hardest to classify.

**Why did your rules fail in some cases?**  
The rule-based system relies only on keyword matching and fails to understand context, intent, and implicit urgency, leading to low action accuracy (41.33%).

**How could an LLM improve this process?**  
An LLM can understand semantic meaning and intent beyond keywords, allowing it to infer urgency and required actions more accurately, improving overall classification.


In [40]:
import pandas as pd

df = pd.read_csv("../data/milestone1_output.csv")
df.head()

,id,sender,subject,body,priority,triage_label,clean_text,keywords,triage
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human,reminder the client meeting is scheduled at t...,"['reminder', 'client', 'meeting', 'scheduled',...",respond_or_act
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond,your invoice of inr is due on please pay to ...,"['invoice', 'inr', 'due', 'please', 'pay', 'av...",respond_or_act
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore,reminder the client meeting is scheduled at t...,"['reminder', 'client', 'meeting', 'scheduled',...",respond_or_act
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond,hello team please find the attached weekly rep...,"['hello', 'team', 'please', 'find', 'attached'...",respond_or_act
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond,hello team please find the attached weekly rep...,"['hello', 'team', 'please', 'find', 'attached'...",respond_or_act


In [41]:
df = pd.read_csv("../data/sample_emails_with_triage_200.csv")

In [42]:
import pandas as pd

df = pd.read_csv("../data/sample_emails_with_triage_200.csv")
df.head()

,id,sender,subject,body,priority,triage_label
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond


In [43]:
# Select 100 test emails

eval_df = df.sample(100,random_state=42)
eval_df = eval_df.reset_index(drop=True)
eval_df.head()

,id,sender,subject,body,priority,triage_label
0,96,no-reply@service.com,Weekly Newsletter,Your order #3634 has been shipped and is expec...,low,ignore
1,16,news@techblog.com,Payment Overdue,"Hi, don't miss our sale with discounts up to 7...",low,notify_human
2,31,news@techblog.com,Weekly Newsletter,Notice: Your account will be locked unless ver...,high,notify_human
3,159,no-reply@service.com,Welcome to Service,Reminder: The client meeting is scheduled at 9...,low,respond
4,129,sales@shop.com,Invoice Due,Your order #6464 has been shipped and is expec...,low,respond


In [44]:

pip install langsmith

In [45]:
from langsmith import Client
client = Client()

In [46]:
judge_prompt = """You are an evaluator. Compare the model output with the ideal answer.

Check:
1. Action correctness
2. Tone correctness

Give score:
1 = correct
0 = incorrect
"""

In [47]:
def evaluate(agent_output, ideal_action, ideal_tone):
    if(
        agent_output["action"] == ideal_action
        and agent_output["tone"] == ideal_tone
    ):
        return 1
    else:
        return 0

In [48]:
agent_output = {
    "action": "notify",
    "tone": "urgent"
}

In [49]:
ideal_action = "notify"
ideal_tone = "urgent"

In [50]:
score = evaluate(agent_output, ideal_action, ideal_tone)
score

1

In [51]:
import pandas as pd

df = pd.read_csv("../data/sample_emails_with_triage_200.csv")
df.head()

,id,sender,subject,body,priority,triage_label
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond


In [52]:
def email_assistant(body):
    text = email_text.lower()
    
    if "urgent" in text or "submit" in text or "deadline" in text:
        return {"action": "notify", "tone": "urgent"}
    
    elif "thank you" in text or "thanks" in text:
        return {"action": "ignore", "tone": "polite"}
    
    else:
        return {"action": "respond", "tone": "neutral"}

In [53]:
# Run the evaluation on sample data

scores = []

for _, row in df.iterrows():
    prediction = email_assistant(row["body"])
    score = evaluate(
        prediction,
        row["ideal_action"],
        row["ideal_tone"]
    )
    scores.append(score)

accuracy = (sum(scores)/len(scores))*100
accuracy

NameError: name 'email_text' is not defined

PART 1: Dataset Preparation

In [54]:
import pandas as pd

df = pd.read_csv("../data/sample_emails_with_triage_200.csv")


In [55]:
def get_ideal_intent(text):
    text = str(text).lower()

    if any(k in text for k in [
        "security alert", "account suspended", "login", "locked", "verification"
    ]):
        return "notify_human"

    if any(k in text for k in [
        "deadline", "mandatory", "submit", "training"
    ]):
        return "notify_human"

    if any(k in text for k in [
        "invoice", "payment due", "pay"
    ]):
        return "respond"

    if any(k in text for k in [
        "meeting", "agenda", "report", "project", "order"
    ]):
        return "respond"

    if any(k in text for k in [
        "congratulations", "won", "sale", "newsletter", "promotion"
    ]):
        return "ignore"

    return "respond"


In [56]:
def get_ideal_tone(text):
    text = str(text).lower()

    if any(k in text for k in [
        "urgent", "deadline", "security", "account", "locked", "suspended"
    ]):
        return "urgent"

    if any(k in text for k in [
        "thank you", "welcome", "newsletter"
    ]):
        return "polite"

    return "neutral"


In [57]:
df["ideal_intent"] = df["body"].apply(get_ideal_intent)
df["ideal_tone"] = df["body"].apply(get_ideal_tone)


In [59]:
df.to_csv("../data/sample_emails_with_triage_200.csv", index=False)
